In [1]:
import kagglehub
path = kagglehub.dataset_download("rmisra/news-category-dataset")

100%|██████████| 26.5M/26.5M [00:00<00:00, 164MB/s]

Extracting files...


In [2]:
import os
import pandas as pd

file=os.listdir(path)[0]
filepath=os.path.join(path,file)
df=pd.read_json(filepath, lines=True)


In [3]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [4]:
## Data Exploration

In [5]:
df.shape

(209527, 6)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [7]:
df=df["short_description"].head(25000)

In [8]:
df.shape

(25000,)

In [9]:
df.duplicated().sum()

np.int64(104)

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
## Preprocessing and Data tokenize

In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [14]:
tokenizer=Tokenizer(oov_token="<OOV>")

In [15]:
tokenizer.fit_on_texts(df)

In [16]:
len(tokenizer.word_index)

29258

In [17]:
data=tokenizer.texts_to_sequences(df)

In [18]:
data[0]

[149,
 383,
 12,
 16,
 9,
 159,
 575,
 3,
 5752,
 352,
 1606,
 68,
 2719,
 49,
 14,
 2,
 15071,
 206,
 4353,
 5,
 2,
 36,
 15072,
 2,
 53,
 56,
 1560,
 8,
 2,
 857]

In [19]:
input_seqeunces=[]
for seq in data:
  if len(seq)<2:
    continue
  for i in range(1,len(seq)):
    n_gram_sequence=seq[:i+1]
    input_seqeunces.append(n_gram_sequence)

In [20]:
input_seqeunces

[[149, 383],
 [149, 383, 12],
 [149, 383, 12, 16],
 [149, 383, 12, 16, 9],
 [149, 383, 12, 16, 9, 159],
 [149, 383, 12, 16, 9, 159, 575],
 [149, 383, 12, 16, 9, 159, 575, 3],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352, 1606],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352, 1606, 68],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352, 1606, 68, 2719],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352, 1606, 68, 2719, 49],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352, 1606, 68, 2719, 49, 14],
 [149, 383, 12, 16, 9, 159, 575, 3, 5752, 352, 1606, 68, 2719, 49, 14, 2],
 [149,
  383,
  12,
  16,
  9,
  159,
  575,
  3,
  5752,
  352,
  1606,
  68,
  2719,
  49,
  14,
  2,
  15071],
 [149,
  383,
  12,
  16,
  9,
  159,
  575,
  3,
  5752,
  352,
  1606,
  68,
  2719,
  49,
  14,
  2,
  15071,
  206],
 [149,
  383,
  12,
  16,
  9,
  159,
  575,
  3,
  5752,
  352,
  1606,
  68,
  2719,
  49,
  14,
  2,
  15

In [21]:
max_length=max([len(i) for i in input_seqeunces])

In [22]:
max_length

44

In [23]:
input_data=pad_sequences(input_seqeunces,padding="pre",maxlen=max_length)

In [24]:
X=input_data[:,:-1]
y=input_data[:,-1]

In [25]:
X.shape

(368084, 43)

In [26]:
y=np.array(y)

In [27]:
## training the model

In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [29]:
vocab_size=len(tokenizer.word_index)+1

In [30]:
vocab_size

29259

In [31]:
max_len=X.shape[1]

In [32]:
max_len

43

In [33]:
model = Sequential([
    Embedding(vocab_size, output_dim=128,mask_zero=False),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(vocab_size, activation="softmax")
])

In [34]:
model.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

In [35]:
model.build(input_shape=(None,max_len))

In [36]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 43, 128)        │     3,745,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 43, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 29259)          │       965,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,772,523 (18.21 MB)

 Trainable params: 4,772,523 (18.21 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test),batch_size=64)

Epoch 1/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 87s 18ms/step - accuracy: 0.0603 - loss: 7.6082 - val_accuracy: 0.0754 - val_loss: 7.3636
Epoch 2/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 79s 17ms/step - accuracy: 0.0851 - loss: 6.9855 - val_accuracy: 0.0936 - val_loss: 7.1510
Epoch 3/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 79s 17ms/step - accuracy: 0.1051 - loss: 6.6466 - val_accuracy: 0.1040 - val_loss: 7.0748
Epoch 4/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 78s 17ms/step - accuracy: 0.1176 - loss: 6.4028 - val_accuracy: 0.1110 - val_loss: 7.0547
Epoch 5/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 78s 17ms/step - accuracy: 0.1257 - loss: 6.2134 - val_accuracy: 0.1141 - val_loss: 7.0812
Epoch 6/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 79s 17ms/step - accuracy: 0.1334 - loss: 6.0568 - val_accuracy: 0.1170 - val_loss: 7.1298
Epoch 7/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 81s 18ms/step - accuracy: 0.1412 - loss: 5.9153 - val_accuracy: 0.1185 - val_loss: 7.1802
Epoch 8/50
4602/4602 ━━━━━━━━━━━━━━━━━━━━ 80s 17ms/step - accuracy: 0.1490 -

In [ ]:
model.save("LSTM_model.h5")


In [ ]:
import pickle
with open("tokenizer.pkl","wb") as f:
  pickle.dump(tokenizer,f)
